# LLaMEA Noisy BBOB Optimization Algorithm Evolution

This notebook acts as the main entry point to prototype, configure, and execute LLaMEA-driven evolution of optimization algorithms on BBOB functions. 

## Environment Configuration
Before running this notebook, ensure your packages are installed using `uv sync --all-extras`.

In [ ]:
# Make sure the src directory is in the path
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import ioh
import llamea

from aad_llm.noisy_bbob import get_noisy_bbob
from aad_llm.prompts import TASK_PROMPT_TEMPLATE, EXAMPLE_PROMPT, FORMAT_PROMPT
from aad_llm.evaluator import Evaluator
import aad_llm.runner as runner

print("Libraries successfully imported!")
print("ioh version:", ioh.__version__)
print("llamea version:", llamea.__version__)

## 1. Sanity Check: Additive Gaussian Noise Wrapper
Let's test the `get_noisy_bbob` function on Problem 1 and plot the distribution of additive noise to confirm it works correctly.

In [ ]:
# Config parameters
problem_id = 1
instance_id = 1
dim = 3
noise_std = 0.05

# Load noisy problem & check its optimum
noisy_func, true_opt = get_noisy_bbob(problem_id, instance_id, dim, noise_std)

print(f"Loaded Problem {problem_id} (Instance {instance_id}, DIM={dim})")
print(f"True target optimum (minimum): {true_opt:.4f}")

# Sample evaluation points
x_test = np.zeros(dim)
y_noisy = [noisy_func(x_test) for _ in range(100)]

print(f"Mean noisy fitness for point {x_test}: {np.mean(y_noisy):.4f}")

# Plot the noise distribution
plt.figure(figsize=(8, 4))
plt.hist(y_noisy, bins=15, edgecolor='black', alpha=0.75, color='#4F46E5')
plt.title(f"Noisy Objective Function Values (Additive Noise Std={noise_std})")
plt.xlabel("Returned Noisy Fitness")
plt.ylabel("Count")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. Inspecting Prompts
Let's verify the dynamic task prompt that gets passed to LLaMEA for problem evolution.

In [ ]:
print("--- TASK PROMPT EXAMPLE (Problem 1) ---")
print(TASK_PROMPT_TEMPLATE.format(problem_id=1))
print("\n--- FORMAT PROMPT ---")
print(FORMAT_PROMPT)

## 3. Remote Server Connection Settings
The client uses the parameters defined at the top of `runner.py`. Inspect those values or override them dynamically below.

In [ ]:
print("Current Base URL:", runner.LLM_BASE_URL)
print("Current Model name:", runner.LLM_MODEL)

# If you want to temporarily override connection parameters for this notebook session:
# runner.LLM_BASE_URL = "https://your-paderborn-server-address/v1"
# runner.LLM_API_KEY = "your-api-key"
# runner.LLM_MODEL = "your-model-name"

## 4. Run Evolution on a Single Problem
Let's run a test evolution loop for 3 iterations on problem 1 (budget 100 evaluations). 
Make sure your target server endpoint is active.

In [ ]:
# Run single problem evolution
try:
    best_sol = runner.run_evolution_for_problem(
        problem_id=1,
        dim=3,
        noise_std=0.05,
        budget=1000,
        iterations=5, # running a brief run for testing
        output_dir="../generated_algorithms",
        log_dir="../logs"
    )
    print("Evolution completed successfully!")
    print("Evolved Class name:", best_sol.name)
    print("Score (Negative Error):", best_sol.score)
except Exception as e:
    print("Could not complete run. Ensure your remote LLM API server is running and accessible.")
    print("Error details:", e)

## 5. Run the Full Experiment
To sequentially evolve optimization algorithms for all 24 BBOB functions, run the following block. Note that this requires access to the remote server and may take a significant amount of time.

In [ ]:
# To run all 24 problems, uncomment the lines below:

# results = runner.run_all_problems(
#     dim=3,
#     noise_std=0.05,
#     budget=1000,
#     iterations=30,
#     output_dir="../generated_algorithms",
#     log_dir="../logs"
# )
# print("Full BBOB suite run completed! Summary of final scores:")
# print(results)